In [1]:
#libraries needed for NLP
import nltk
nltk.download('punkt_tab')
from nltk.stem.lancaster import LancasterStemmer
stemmer = LancasterStemmer()

import tensorflow as tf
import numpy as np
import random
import json

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\shahd\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
file_path = r"intents.json"

with open(file_path, "r", encoding="utf-8") as file:
    intents = json.load(file)

In [3]:
intents

{'intents': [{'context': [''],
   'patterns': ['Hi there',
    'Greetings',
    'Is anyone there?',
    'Hey',
    'Hola',
    'Hello',
    'Good day'],
   'responses': ['Hi there!',
    'Good to see you again',
    'Hi there, how can I help?'],
   'tag': 'greeting'},
  {'context': [''],
   'patterns': ['How are you',
    "What's up?",
    'How is it going?',
    'How are you today?'],
   'responses': ['Hello, thanks for asking',
    'Good to see you again',
    'Hi there, how can I help?'],
   'tag': 'how_are_you'},
  {'context': [''],
   'patterns': ['Who are you',
    "What's your name?",
    'What are you',
    'Who is this'],
   'responses': ["I'm Travel Buddy",
    'My name is Travel Buddy',
    "Thanks for asking, I'm Travel Buddy!"],
   'tag': 'who_are_you'},
  {'context': [''],
   'patterns': ['Bye',
    'See you later',
    'Goodbye',
    'Nice chatting to you, bye',
    'Till next time'],
   'responses': ['See you!', 'Have a nice day', 'Bye! Come back again soon.'],
   'tag'

In [4]:
words = []
classes = []
documents = []
ignore = ['?']

for intent in intents['intents']:
  for pattern in intent['patterns']:
    w = nltk.word_tokenize(pattern)
    words.extend(w)
    documents.append((w, intent['tag']))
    if intent['tag'] not in classes:
      classes.append(intent['tag'])

In [5]:
words = [stemmer.stem(w.lower()) for w in words if w not in ignore]
words = sorted(list(set(words)))
classes = sorted(list(set(classes)))

print(len(documents), "documents")
print(len(classes), "classes", classes)
print(len(words), "uniques stemmed words", words)

35 documents
7 classes ['goodbye', 'greeting', 'how_are_you', 'options', 'thanks', 'travel_tips', 'who_are_you']
57 uniques stemmed words ["'s", ',', 'a', 'anyon', 'ar', 'awesom', 'be', 'bye', 'can', 'chat', 'could', 'day', 'do', 'for', 'get', 'giv', 'going', 'good', 'goodby', 'greet', 'hello', 'help', 'hey', 'hi', 'hol', 'how', 'i', 'is', 'it', 'lat', 'me', 'mod', 'nam', 'next', 'nic', 'off', 'op', 'provid', 'see', 'support', 'thank', 'that', 'ther', 'thi', 'til', 'tim', 'tip', 'to', 'today', 'travel', 'up', 'want', 'what', 'who', 'with', 'yo', 'you']


In [6]:
training = []
output = []

output_empty = [0] * len(classes)
for doc in documents:

  bag = []
  pattern_words = doc[0]
  pattern_words = [stemmer.stem(word.lower()) for word in pattern_words]
  for w in words:
    bag.append(1) if w in pattern_words else bag.append(0)

  output_row = list(output_empty)
  output_row[classes.index(doc[1])] = 1

  training.append([bag, output_row])

random.shuffle(training)
training = np.array(training, dtype=object)

train_x = list(training[:,0])
train_y = list(training[:,1])

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

train_x = np.array(train_x, dtype=np.float32)
train_y = np.array(train_y, dtype=np.float32)

model = Sequential([
    Dense(10, activation='relu', input_shape=(len(train_x[0]),)),
    Dense(10, activation='relu'),
    Dense(len(train_y[0]), activation='softmax')
])

model.compile(
    optimizer=SGD(learning_rate=0.01, momentum=0.9),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_x,
    train_y,
    epochs=1000,
    batch_size=8,
    verbose=1
)

model.save("model.keras")


Epoch 1/1000


5/5 [==============================] - 1s 3ms/step - loss: 1.9257 - accuracy: 0.1143
Epoch 2/1000
5/5 [==============================] - 0s 2ms/step - loss: 1.9186 - accuracy: 0.2000
Epoch 3/1000
5/5 [==============================] - 0s 2ms/step - loss: 1.9063 - accuracy: 0.2571
Epoch 4/1000
5/5 [==============================] - 0s 2ms/step - loss: 1.8959 - accuracy: 0.3143
Epoch 5/1000
5/5 [==============================] - 0s 2ms/step - loss: 1.8806 - accuracy: 0.3429
Epoch 6/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.8673 - accuracy: 0.3143
Epoch 7/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.8532 - accuracy: 0.3143
Epoch 8/1000
5/5 [==============================] - 0s 2ms/step - loss: 1.8368 - accuracy: 0.3429
Epoch 9/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.8188 - accuracy: 0.3714
Epoch 10/1000
5/5 [==============================] - 0s 2ms/step - loss: 1.8012 - accuracy: 0.3714
Epoch 11/1000
5/

In [8]:
import pickle
pickle.dump( { 'words': words, 'classes':classes, 'train_x':train_x, 'train_y':train_y}, open("training_data", "wb"))

In [9]:
data = pickle.load( open("training_data", "rb"))
words = data['words']
classes = data['classes']
train_x = data['train_x']
train_y = data['train_y']

In [10]:
with open('intents.json') as json_data:
  intents = json.load(json_data)

In [11]:
from tensorflow.keras.models import load_model

model = load_model("model.keras")

In [12]:
def clean_up_sentence(sentence):

  sentence_words = nltk.word_tokenize(sentence)

  sentence_words = [stemmer.stem(word.lower()) for word in sentence_words]
  return sentence_words

def bow(sentence, words, show_details=False):
  sentence_words = clean_up_sentence(sentence)
  bag = [0]*len(words)
  for s in sentence_words:
        for i,w in enumerate(words):
            if w == s:
                bag[i] = 1
                if show_details:
                    print ("found in bag: %s" % w)
  return(np.array(bag))

In [13]:
ERROR_THRESHOLD = 0.30
def classify(sentence):

    results = model.predict(
    np.array([bow(sentence, words)], dtype=np.float32),
    verbose=0)[0]
    results = [[i,r] for i,r in enumerate(results) if r>ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)
    return_list = []
    for r in results:
        return_list.append((classes[r[0]], r[1]))
    return return_list

def response(sentence, userID='123', show_details=False):
    results = classify(sentence)
    if results:
        while results:
            for i in intents['intents']:
                if i['tag'] == results[0][0]:
                    return print(random.choice(i['responses']))

            results.pop(0)

In [14]:
response('')

Good to see you again


In [15]:
response('How are you')

Good to see you again


In [16]:
response('any tips for my trip')

Have a nice day


In [17]:
response('tips')

Bye! Come back again soon.


In [18]:
response('travel tips')

Always pack extra socks!


In [19]:
response('')

Hi there, how can I help?


In [20]:
response('open travel tips')

Remember: Don't beat up your feet! Bring sensible, comfortable, broken-in shoes.


In [21]:
response('travel tips')

Always pack extra socks!


In [22]:
response('Available packages')

Hi there!


In [23]:
response('Thanks')

Any time!


In [24]:
response('How are you')

Hi there, how can I help?


In [25]:
response('how are you doing')

Hello, thanks for asking


In [26]:
response('travel tips')

Don't forget to bring a towel! No, seriously!


In [27]:
response('what are you doing')

Thanks for asking, I'm Travel Buddy!


In [28]:
response('')

Hi there, how can I help?


In [29]:
response('travel tips')

Here's a tip: When traveling to a new city, be on the lookout for free walking tours!


In [30]:
response('hi')

Good to see you again


In [31]:
response('hello there')

Good to see you again
